# validation_and_helpers

Two unrelated things that are both *support* code, never the computation itself:

1. **the pure-numpy reference** — `populations_enhanced` / `populations_enhanced_memory` redo the
   propagation as plain matrix algebra so the Qiskit result can be checked against it. **No circuit,
   no simulator.** `propagate_dilated` builds the same Sz.-Nagy unitary the circuit uses and then just
   multiplies it onto a vector.
2. **the run cache** — `save_run` / `load_run` move one complete run between `main.ipynb` (which
   computes) and `results.ipynb` (which only plots), and `channel_of` rebuilds the Qiskit channel that
   is too large to store.

| notebook | role |
|---|---|
| `grid.ipynb` | building blocks the real grid uses |
| `circuit.ipynb` | **the quantum grid** — dilated `UnitaryGate` on Aer |
| `main.ipynb` | runs everything, writes `data/run_N*.npz` |
| `results.ipynb` | reads that file and plots — computes nothing |
| **this file** | reference implementation + the load/save glue |

In [ ]:
import numpy as np
import import_ipynb                                  # lets `import` read .ipynb modules
from grid import vec, unvec, enhanced_kraus, transfer_tensors, companion_propagator, load_maps
from circuit import DilatedChannel                   # only to rebuild a channel for a cached run

### Sz - Nagy - Dilation
Baut die Matrix $ U= \begin{pmatrix} A_s & B \\ C & -A_s^\dagger \end{pmatrix} = \begin{pmatrix}E_s & \sqrt{I-E_sE_s^\dagger}\\ \sqrt{I-E_s^\dagger E_s} & -E_s^\dagger\end{pmatrix}$

In [ ]:
def sz_nagy_dilation(E, s=None):
    """Unitary U = [[E/s, B], [C, -(E/s)^dag]] with B=(I-EE^dag/s^2)^(1/2),
    C=(I-E^dag E/s^2)^(1/2): one block-encoded unitary for the (generally
    non-CP) operator E, exactly the thesis dilation trick generalised.
    Returns (U, s); applying U, keeping the top block and renormalising
    reproduces E/s."""

    if s is None:
        s = np.linalg.norm(E, 2) * (1.0 + 1e-12)

    Es = E / s
    I = np.eye(Es.shape[0])
    B = sqrtm(I - Es @ Es.conj().T)
    C = sqrtm(I - Es.conj().T @ Es)
    B, C = 0.5 * (B + B.conj().T), 0.5 * (C + C.conj().T)
    
    return np.block([[Es, B], [C, -Es.conj().T]]), float(s)

Propagier den Zustand $x$ mit der dilated Matrix $U$ für alle gewünschten Zeiten.

In [ ]:
def propagate_dilated(E, x0, n_steps):
    """THE GRID for a non-CP one-step operator: dilate E/s into a single
    unitary U once, then run the thesis loop -- apply the SAME U to the
    (normalised, ancilla-padded) register, read the top block out,
    renormalise, feed it back in as the next input. No operator is ever
    rebuilt. Returns (trajectory (n_steps+1, dim), U, s)."""
    U, s = sz_nagy_dilation(E)      # berechnet dilation of E/s
    n = E.shape[0]                  # Bestimmt die Dimension des physikalischen Systems.
    x = np.asarray(x0, complex)     # x0 ist der state vector, der in das System eingegeben wird
    traj = [x.copy()]               # Legt eine Liste an, um den Zustand des Systems nach jedem Zeitschritt zu speichern
    
    for _ in range(n_steps):
        nrm = np.linalg.norm(x)
        y = U @ np.concatenate([x / nrm, np.zeros(n)])   # propagiert Zustand x mit U
        x = y[:n] * (s * nrm)       # System wird ausgelesen, indem nur die ersten n Einträge behalten werden und der Vektor wird renormalisiert                           
        traj.append(x.copy())

    return np.array(traj), U, s

In [ ]:
def populations_enhanced_memory(maps_in, rho0, K, n_steps, via="grid"):
    """Enhanced algorithm with memory. Learn T_1..T_K from the short-time
    maps and build the ONE fixed companion operator E, then propagate by
    re-applying it. n_steps may (and normally does) run far past the last
    time contained in the input maps -- beyond that the trajectory is pure
    prediction, at no further classical cost. The startup history
    rho_n = L_n rho_0 (n < K) comes from the same learning data.

    maps_in : map file path, loaded dict, or a raw (K+1, D, D) array.
    via     : "grid"   -- propagate through the dilated unitary (the actual
                          enhanced-algorithm quantum grid; the default),
              "direct" -- plain classical iteration X -> E X (reference).

    Returns dict: pops (n_steps+1, d), rho, T_norms, E, K, and for via="grid"
    also U, s (the single reused unitary and its scale)."""
    if isinstance(maps_in, str):
        maps_in = load_maps(maps_in)

    maps = maps_in["maps"] if isinstance(maps_in, dict) else np.asarray(maps_in)
    D = maps.shape[1]
    d = int(round(np.sqrt(D)))
    
    if K > len(maps) - 1:
        raise ValueError(f"K={K} needs maps up to index K (have {len(maps)-1})")

    T = transfer_tensors(maps, K)
    E = companion_propagator(T)                    # computed ONCE

    v0 = vec(np.asarray(rho0, complex))
    hist = [maps[n] @ v0 for n in range(min(K, n_steps + 1))]
    X0 = np.concatenate(hist[::-1])                # newest state = first block
    out = dict(T_norms=np.array([np.linalg.norm(Tm, 2) for Tm in T]), E=E, K=K)

    if via == "grid":
        Xs, U, s = propagate_dilated(E, X0, max(n_steps + 1 - K, 0))
        out.update(U=U, s=s)
        states = hist + [X[:D] for X in Xs[1:]]
    elif via == "direct":
        X, states = X0, list(hist)
        for _ in range(K, n_steps + 1):
            X = E @ X
            states.append(X[:D].copy())
    else:
        raise ValueError("via must be 'grid' or 'direct'")

    traj = np.array([unvec(v, d) for v in states[:n_steps + 1]])
    out.update(pops=np.real(np.einsum("tii->ti", traj)), rho=traj)
    return out

Diese Funktion wird nur verwedet um den enhanced algorithmus zu validieren. Dabei wird eine Rechnung die wirklich auf dem Grid durchgeführt wurde mit dieser Funktion hier verglichen die das ganze mit normaler Matrix-Multiplikation rechnet ohne ein Grid zu verwenden.

In [ ]:
def populations_enhanced(L_dt, d, rho0, n_steps):
    """Build M_k(dt) once from the one-step map L(dt), then re-apply it
    n_steps times. n_steps is unbounded -- no further classical work.
    Returns dict: pops (n_steps+1, d), rho, kraus_dt, n_kraus, min_choi_eig."""
    
    kraus, min_eig = enhanced_kraus(L_dt, d)    # compute the Kraus operators once
    rho = np.asarray(rho0, complex)             # initial state
    traj = [rho.copy()]                         # store the trajectory of density matrices, starting with the initial state

    for _ in range(n_steps):                     # re-apply the SAME channel
        rho = sum(M @ rho @ M.conj().T for M in kraus)
        traj.append(rho)

    traj = np.array(traj)
    return dict(pops=np.real(np.diagonal(traj, axis1=1, axis2=2)), rho=traj,
                kraus_dt=kraus, n_kraus=len(kraus), min_choi_eig=min_eig)

## The run cache

`main.ipynb` computes one run and writes it; `results.ipynb` reads it. The Qiskit channel objects are
the only thing left out — a `DilatedChannel` carries the full $2^n\times2^n$ unitary (268 MB for the
7-site HEOM register), so it is rebuilt on demand from $E$ or the Kraus operators instead.

In [ ]:
def save_run(path, run):
    """Write one complete run (a plain dict) to a single .npz."""
    np.savez_compressed(path, run=np.array(run, dtype=object))


def load_run(path):
    """Read it back."""
    return np.load(path, allow_pickle=True)["run"].item()


def channel_of(r):
    """The DilatedChannel a method re-applies, rebuilt from the cached result
    (one sqrtm; the stored run deliberately does not carry the unitary)."""
    if "E" in r:
        return DilatedChannel(r["E"], "E")
    M = r["kraus_dt"][0]                                  # Lindblad: branch 0
    return DilatedChannel(np.kron(M.conj(), M), "M0")